In [5]:
from transformers import AutoTokenizer
from SMOLL2_135M_imitation_model import GQA_LLM
import torch

# Load model directly
tokenizer =AutoTokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")

dim = 576
num_layers = 30
hidden_dim = 1536
model = GQA_LLM(49152, dim, num_layers=30, hidden_dim=1536,n_heads = 8,
num_kv_heads = 2)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
import torch

In [8]:
# Load the checkpoint
checkpoint = torch.load("/content/checkpoints/checkpoint_final.pt")

<ipython-input-8-edbdeab20edf>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/content/checkpoints/checkpoint_final.pt")


In [10]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])  # If you have an optimizer in the checkpoint

In [11]:
# Optionally move model to GPU if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GQA_LLM(
  (embedding): Embedding(49152, 576)
  (layers): ModuleList(
    (0-29): 30 x TransformerBlock(
      (attn): SelfAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=True)
        (k_proj): Linear(in_features=576, out_features=144, bias=True)
        (v_proj): Linear(in_features=576, out_features=144, bias=True)
        (o_proj): Linear(in_features=576, out_features=576, bias=True)
        (rotary_emb): RotaryEmbedding()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=576, out_features=1536, bias=True)
        (fc2): Linear(in_features=1536, out_features=576, bias=True)
      )
      (norm1): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
    )
  )
  (norm): LayerNorm((576,), eps=1e-05, elementwise_affine=True)
  (output_layer): Linear(in_features=576, out_features=49152, bias=True)
)

In [12]:
from train import load_input_file_dataset,get_dataloader
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
from torch.amp import autocast  # Correct import

scaler = torch.cuda.amp.GradScaler()

# Modified to assign dataset to the first element of returned value
dataset, tokenizer = load_input_file_dataset(seq_length=750) # assign the first element (input_ids) to dataset

dataloader = get_dataloader(dataset, batch_size=32)

# Assuming `device` is already set to either "cuda" or "cpu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to the correct device
model = model.to(device)

# Ensure optimizer is created after moving the model to the device
optimizer = optim.Adam(model.parameters(), lr=0.001)

step = 0
model.train()
while step < 50:
    for batch in dataloader:

        # Ensure all tensors are on the correct device
        input_ids = batch[0].to(device)
        labels = input_ids[:, 1:].contiguous().to(device)
        inputs = input_ids[:, :-1].contiguous().to(device)

        optimizer.zero_grad()

        criterion = nn.CrossEntropyLoss()

        # Updated autocast usage
        with autocast(device_type=device.type):  # No need for `device_type`
            outputs = model(inputs)
            loss = criterion(outputs.view(-1, 49152), labels.view(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        step += 1
        print(f"The step number is step {step}: {loss.item()}")

Loaded 341093 tokens
1 epoch = 28 batches


Steps in current interval:  33%|███▎      | 167/500 [01:08<02:15,  2.46it/s]

KeyboardInterrupt: 

In [17]:
import torch

inputs = tokenizer("How are you?", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

with torch.no_grad():
    outputs = model(input_ids)  # This returns logits, not token IDs

# The model's output might be the logits directly.
# Try using 'outputs' instead of 'outputs.logits'
predicted_token_ids = torch.argmax(outputs, dim=-1)  # Get most likely token IDs

# Decode the predicted token IDs
print(tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True))

 ours applicant Inform Dias
